# 01 — Scene Sanity Check

Verify the Mitsuba scene loads correctly:
- bounding box matches the Gazebo world (0,0,0)→(5,4,3)
- all 6 surfaces present
- ITU concrete material registered
- visual render looks like a rectangular room

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import sionna
import sionna.rt as rt
from sionna.rt import Camera

print(f"Sionna version: {sionna.__version__}")
assert sionna.__version__ == "1.2.1", f"Expected Sionna 1.2.1, got {sionna.__version__}"
print("Version OK")

## Load scene

In [ ]:
scene_path = os.path.abspath("../scene/room.xml")
print(f"Scene XML: {scene_path}")
scene = rt.load_scene(scene_path)
print(f"Objects     : {list(scene.objects.keys())}")
print(f"Materials   : {list(scene.radio_materials.keys())}")

## Bounding box assertion

The room interior spans exactly (0,0,0) → (5,4,3),
matching the Gazebo world coordinate frame.

In [ ]:
bbox = scene.mi_scene.bbox()
b_min = np.array(bbox.min)
b_max = np.array(bbox.max)

print(f"BBox min : {b_min}")
print(f"BBox max : {b_max}")

TOL = 0.01   # 1 cm tolerance
expected_min = np.array([0.0, 0.0, 0.0])
expected_max = np.array([5.0, 4.0, 3.0])

assert np.allclose(b_min, expected_min, atol=TOL), (
    f"BBox min mismatch: got {b_min}, expected {expected_min}.\n"
    "Check PLY vertex coordinates in scene/gen_meshes.py")
assert np.allclose(b_max, expected_max, atol=TOL), (
    f"BBox max mismatch: got {b_max}, expected {expected_max}.\n"
    "Check PLY vertex coordinates in scene/gen_meshes.py")

print("\nBOUNDING BOX ASSERTION PASSED")

## Visual renders

Two camera angles: top-down and perspective corner view.
All six surfaces (floor, ceiling, 4 walls) should be visible.

In [ ]:
import os
RENDERS = os.path.abspath("../renders")
os.makedirs(RENDERS, exist_ok=True)

# Top-down view
cam_top = Camera(position=[2.5, 2.0, 12.0], look_at=[2.5, 2.0, 1.5])
bmp_top = scene.render(camera=cam_top, num_samples=128, resolution=(600, 480),
                       return_bitmap=True)
img_top = np.array(bmp_top)
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img_top); ax.set_title("Top-down view (looking -Z)"); ax.axis("off")
plt.tight_layout()
out = os.path.join(RENDERS, "01_top_down.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

In [ ]:
# Perspective corner view
cam_persp = Camera(position=[-3.0, -2.0, 5.0], look_at=[2.5, 2.0, 1.5])
bmp_persp = scene.render(camera=cam_persp, num_samples=128, resolution=(600, 480),
                         return_bitmap=True)
img_persp = np.array(bmp_persp)
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img_persp); ax.set_title("Perspective corner view"); ax.axis("off")
plt.tight_layout()
out = os.path.join(RENDERS, "01_perspective.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

All assertions passed. Scene is geometrically correct. Proceed to `02_single_point_test.ipynb`.